# UniBots UK 2026 — YOLO11n Ball Detection Training

Rulebook: **Unibots 2026 V1.2** (§4 Arena + §4.3 Balls)

**Classes:**
| ID | Name | Rulebook spec |
|---|---|---|
| 0 | `ping_pong_ball` | 40 mm diameter, **orange**, may have markings (§4.3.2) |
| 1 | `bearing` | **20 mm** diameter, polished carbon steel, silver/grey (§4.3.3) |
| 2 | `robot` | competition robots, max 300 mm cube (§3.3) |

**Arena (§4.1):**
- Floor: **white painted**, 2000×2000 mm
- Walls: **black**, ~150 mm tall
- Scoring zone walls: Yellow (N) / Orange (E) / Purple (S) / Green (W)
- Black 19 mm tape lines: 4 rounded-square tracks on floor (§4.4)
- 40 balls total: **16 ping-pong + 24 bearings** (§4.3.1)

**Speed targets (NCNN, RPi4/BananaPi/NanoPi, 4 threads):**
| Input | Est. FPS |
|---|---|
| 256×256 | ~30–35 fps ✅ |
| 320×320 | ~20–25 fps |

Run on Colab: **Runtime → Change runtime type → T4 GPU**

## 1. Environment Check

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'NONE — enable GPU runtime')

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics roboflow albumentations Pillow

In [ ]:
import ultralytics, torch
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 3. Dataset Configuration

**Datasets combined:**
| # | Source | Content | How |
|---|---|---|---|
| 1 | BARF/Dylan | ping pong + bearing | Roboflow API |
| 2 | SDP Pong Pickup Pro | ping pong | Roboflow API |
| 3 | ECE4078 Ball Bearing v4 | bearings | Roboflow API |
| 4 | Data Conversion | ping pong | Roboflow API |
| 5 | Xi-X Ultralytics Hub | ping pong (2 259 imgs) | NDJSON auto-converter |
| 6 | **Synthetic** | rulebook-accurate arena | PIL generator, CPU-only |

Free Roboflow account: https://roboflow.com → Settings → API Key

**For dataset 5:** export NDJSON from `platform.ultralytics.com/xi-x/datasets/ping-pong-ball` → upload to Colab as `/content/ping-pong-ball.ndjson` → set `NDJSON_PATH` in cell below.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURE — edit this cell, then run top-to-bottom
# ─────────────────────────────────────────────────────────────────────────────
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"

# Ultralytics Hub NDJSON: export from platform.ultralytics.com → upload to Colab.
# Set to None to skip.
NDJSON_PATH = "/content/ping-pong-ball.ndjson"

# class_map values: 0=ping_pong_ball  1=bearing  2=robot  -1=discard
#
# Keys MUST exactly match (or match lowercased) class names in each data.yaml.
# Exact names confirmed by diagnostic cell (run after downloading):
#   dataset_0 (BARF/Dylan)       → ['sports_ball']
#   dataset_1 (SDP Pong)         → ['Ping-Pong ball']
#   dataset_2 (ECE4078 Bearing)  → ['0']          ← string "0", not int
#   dataset_3 (Data Conversion)  → ['ping pong ball']
#   Hub NDJSON                   → ['ball']        ← from ndjson header class_names
DATASETS = [
    # ── 1. BARF/Dylan ────────────────────────────────────────────────────────
    # Confirmed class name: 'sports_ball'
    {
        "workspace": "dylans-workspace-3init",
        "project":   "my-first-project-sbk3a",
        "version":   1,
        "class_map": {
            "sports_ball": 0, "sports ball": 0,      # confirmed exact names
            "ping-pong": 0, "ping_pong_ball": 0, "pingpong": 0, "ball": 0,
            "bearing": 1, "steel-ball": 1, "ball-bearing": 1,
            "robot": 2,
        }
    },
    # ── 2. SDP Pong Pickup Pro ───────────────────────────────────────────────
    # Confirmed class name: 'Ping-Pong ball'  (capital P, space, lowercase b)
    {
        "workspace": "sdp-pongpickup-pro",
        "project":   "ping-pong-ball-detection-e04ql",
        "version":   1,
        "class_map": {
            "Ping-Pong ball": 0,                     # confirmed exact name
            "ping-pong ball": 0,                     # lowercase fallback
            "ping-pong-ball": 0, "ping_pong_ball": 0,
            "pingpong": 0, "ball": 0,
        }
    },
    # ── 3. ECE4078 Ball Bearing Detection v4 ─────────────────────────────────
    # Confirmed class name: '0'  (the string "0", not an integer)
    {
        "workspace": "ece4078-bkprr",
        "project":   "ball-bearing-detection-v4",
        "version":   1,
        "class_map": {
            "0": 1,                                  # confirmed exact name — string "0"
            "ball-bearing": 1, "ball_bearing": 1, "bearing": 1,
            "Bearing": 1, "steel-ball": 1, "ball": 1,
        }
    },
    # ── 4. Data Conversion ping pong ─────────────────────────────────────────
    # Confirmed class name: 'ping pong ball'  (spaces, no hyphens)
    {
        "workspace": "data-conversion-givrq",
        "project":   "ping-pong-balls",
        "version":   1,
        "class_map": {
            "ping pong ball": 0,                     # confirmed exact name
            "ping-pong-ball": 0, "ping_pong_ball": 0,
            "pingpong": 0, "ball": 0,
        }
    },
]

TARGET_CLASSES         = ['ping_pong_ball', 'bearing', 'robot']
TRAIN_IMGSZ            = 640
TRAIN_EPOCHS           = 150
TRAIN_BATCH            = -1
EXPORT_IMGSZ_PRIMARY   = 256   # ~30fps on RPi4 Cortex-A72
EXPORT_IMGSZ_SECONDARY = 320   # ~22fps, higher accuracy
N_SYNTHETIC_IMAGES     = 1000  # increase to 3000–5000 for better edge coverage

print('Config OK. Classes:', TARGET_CLASSES)

In [ ]:
ULTRALYTICS_HUB_DIR = None  # e.g. '/content/ultralytics_hub_pingpong'
ULTRALYTICS_HUB_CLASS_MAP = {
    "ping-pong-ball": 0, "ping_pong_ball": 0, "ping-pong": 0,
    "ping_pong": 0, "ball": 0,
}

In [ ]:
import os, shutil, yaml, zipfile
from pathlib import Path
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
downloaded_dirs = []


def try_download(workspace, project, version, location):
    """
    Download with fallbacks for corrupt zips.

    Some datasets only have certain export formats prepared on Roboflow.
    'yolov11' can produce a bad zip; fall back to 'yolov8' then 'yolov5'
    (all use identical YOLO label structure — fully interchangeable here).

    Also probes version numbers 1–4 automatically, since projects with
    'v4' in their *name* sometimes only have version 1 published on the API.
    Specified version is tried first.
    """
    proj = rf.workspace(workspace).project(project)
    formats  = ['yolov11', 'yolov8', 'yolov5']
    versions = [version] + [v for v in [1, 2, 3, 4] if v != version]

    for v in versions:
        for fmt in formats:
            loc = f'{location}_v{v}_{fmt}'
            try:
                ds = proj.version(v).download(fmt, location=loc)
                # Verify extraction succeeded — SDK sometimes silently leaves
                # a corrupt zip without raising an exception
                yaml_files = list(Path(ds.location).rglob('data.yaml'))
                if not yaml_files:
                    raise RuntimeError('data.yaml not found after download')
                print(f'  OK  v{v}/{fmt} → {ds.location}')
                return ds.location
            except Exception as e:
                msg = str(e)
                if any(k in msg.lower() for k in
                       ['zip', 'central-directory', 'central directory',
                        'data.yaml', 'eocd', 'bad zip', 'extract']):
                    print(f'  corrupt zip v{v}/{fmt} — trying next format...')
                elif '404' in msg or 'not found' in msg.lower():
                    print(f'  v{v} not found — trying next version...')
                    break  # other formats won't help for a missing version
                else:
                    print(f'  error v{v}/{fmt}: {msg[:120]}')
    return None


for i, ds in enumerate(DATASETS):
    print(f"\n[{i+1}/{len(DATASETS)}] {ds['workspace']}/{ds['project']}")
    loc = try_download(
        ds['workspace'], ds['project'], ds['version'],
        f'/content/dataset_{i}'
    )
    if loc:
        # data.yaml may sit one level inside the download dir
        yaml_files = list(Path(loc).rglob('data.yaml'))
        root = str(yaml_files[0].parent)
        downloaded_dirs.append((root, ds['class_map']))
    else:
        print(f'  SKIPPED — all versions/formats failed.')
        print(f'  Fix: open https://universe.roboflow.com/{ds["workspace"]}/{ds["project"]}')
        print(f'       find available version numbers, update DATASETS[{i}]["version"], re-run.')

if ULTRALYTICS_HUB_DIR and Path(ULTRALYTICS_HUB_DIR).exists():
    downloaded_dirs.append((ULTRALYTICS_HUB_DIR, ULTRALYTICS_HUB_CLASS_MAP))
    print(f'\nAdded Ultralytics Hub dataset: {ULTRALYTICS_HUB_DIR}')

print(f'\n{len(downloaded_dirs)}/{len(DATASETS)} datasets ready for merging.')

In [ ]:
# ── Diagnostic: verify actual class names in each downloaded dataset ──────────
# Run this after the download cell to confirm class_map keys match.
# If a dataset shows names you didn't expect, update DATASETS[i]["class_map"] above
# and re-run from the merge cell (§5) — no need to re-download.
import yaml
from pathlib import Path
print('Class names found in downloaded datasets:')
for p in sorted(Path('/content').rglob('data.yaml')):
    try:
        names = yaml.safe_load(p.read_text()).get('names', [])
        print(f'  {p.parent.name} → {names}')
    except Exception as e:
        print(f'  {p.parent.name} → ERROR: {e}')

import os, shutil, yaml, zipfile
from pathlib import Path
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
downloaded_dirs = []


def try_download(workspace, project, version, location):
    """
    Download with fallbacks for corrupt zips.
    Tries formats yolov11→yolov8→yolov5 (identical label structure).
    Probes versions 1–4; specified version first.
    """
    proj = rf.workspace(workspace).project(project)
    formats  = ['yolov11', 'yolov8', 'yolov5']
    versions = [version] + [v for v in [1, 2, 3, 4] if v != version]

    for v in versions:
        for fmt in formats:
            loc = f'{location}_v{v}_{fmt}'
            try:
                ds = proj.version(v).download(fmt, location=loc)
                yaml_files = list(Path(ds.location).rglob('data.yaml'))
                if not yaml_files:
                    raise RuntimeError('data.yaml not found after download')
                print(f'  OK  v{v}/{fmt} → {ds.location}')
                return ds.location
            except Exception as e:
                msg = str(e)
                if any(k in msg.lower() for k in
                       ['zip', 'central-directory', 'central directory',
                        'data.yaml', 'eocd', 'bad zip', 'extract']):
                    print(f'  corrupt zip v{v}/{fmt} — trying next format...')
                elif '404' in msg or 'not found' in msg.lower():
                    print(f'  v{v} not found — trying next version...')
                    break
                else:
                    print(f'  error v{v}/{fmt}: {msg[:120]}')
    return None


for i, ds in enumerate(DATASETS):
    print(f"\n[{i+1}/{len(DATASETS)}] {ds['workspace']}/{ds['project']}")
    loc = try_download(
        ds['workspace'], ds['project'], ds['version'],
        f'/content/dataset_{i}'
    )
    if loc:
        yaml_files = list(Path(loc).rglob('data.yaml'))
        root = str(yaml_files[0].parent)
        downloaded_dirs.append((root, ds['class_map']))
    else:
        print(f'  SKIPPED — all versions/formats failed.')
        print(f'  Fix: open https://universe.roboflow.com/{ds["workspace"]}/{ds["project"]}')
        print(f'       check available versions, update DATASETS[{i}]["version"], re-run.')

print(f'\n{len(downloaded_dirs)}/{len(DATASETS)} Roboflow datasets downloaded.')

In [ ]:
# ── Ultralytics Hub NDJSON → YOLO converter ───────────────────────────────────
# How to get the NDJSON:
#   1. Go to: https://platform.ultralytics.com/xi-x/datasets/ping-pong-ball
#   2. Export → NDJSON  (or: three-dot menu → Download)
#   3. Upload the .ndjson file to Colab (Files panel → Upload)
#   4. Run this cell — images download automatically (2 259 images, ~200 MB)
# Set NDJSON_PATH=None in the config cell to skip.
# ─────────────────────────────────────────────────────────────────────────────
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
import urllib.request

_NDJSON_OUT = "/content/ping_pong_yolo"

if NDJSON_PATH and Path(NDJSON_PATH).exists():
    def _parse_ndjson(path):
        header, images = None, []
        with open(path) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                r = json.loads(line)
                if r.get("type") == "dataset":  header = r
                elif r.get("type") == "image":  images.append(r)
        return header or {"class_names": {"0": "ball"}}, images

    def _dl(url, dest):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=30) as r: dest.write_bytes(r.read())
            return True
        except Exception as e:
            print(f"  ✗ {dest.name}: {e}"); return False

    header, images = _parse_ndjson(NDJSON_PATH)
    _cls = [v for _, v in sorted(header["class_names"].items(), key=lambda kv: int(kv[0]))]
    _out = Path(_NDJSON_OUT)
    _splits = Counter(img.get("split", "train") for img in images)
    print(f"Hub NDJSON: {len(images)} images → {dict(_splits)}, classes: {_cls}")

    # NDJSON layout: images/{split}/ and labels/{split}/
    # (different from Roboflow {split}/images/ — find_split handles both)
    for s in _splits:
        (_out / "images" / s).mkdir(parents=True, exist_ok=True)
        (_out / "labels" / s).mkdir(parents=True, exist_ok=True)

    # Write label files
    for img in images:
        lp = _out / "labels" / img.get("split", "train") / (Path(img["file"]).stem + ".txt")
        boxes = img.get("annotations", {}).get("boxes", [])
        lp.write_text("\n".join(
            f"{int(b[0])} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}" for b in boxes
        ))
    print(f"✓ {len(images)} label files written")

    # Download images (parallel, skip already-downloaded)
    _tasks = [
        (img["url"], _out / "images" / img.get("split", "train") / img["file"])
        for img in images
        if not (_out / "images" / img.get("split", "train") / img["file"]).exists()
    ]
    if _tasks:
        print(f"Downloading {len(_tasks)} images (16 threads)…")
        _ok = 0
        with ThreadPoolExecutor(max_workers=16) as _pool:
            _futs = {_pool.submit(_dl, url, dest): dest for url, dest in _tasks}
            for _i, _fut in enumerate(as_completed(_futs), 1):
                if _fut.result(): _ok += 1
                if _i % 200 == 0 or _i == len(_tasks):
                    print(f"  {_i}/{len(_tasks)}", end="\r")
        print(f"\n✓ {_ok}/{len(_tasks)} images downloaded")
    else:
        print("Images already cached — skipping download.")

    # Write data.yaml (named data.yaml so load_class_names finds it)
    import yaml as _yaml
    (_out / "data.yaml").write_text(_yaml.dump({
        "path": str(_out.resolve()), "train": "images/train", "val": "images/val",
        "nc": len(_cls), "names": _cls,
    }, default_flow_style=False))

    downloaded_dirs.append((_NDJSON_OUT, {
        "ball": 0,                              # confirmed class name in this NDJSON
        "ping-pong-ball": 0, "ping_pong_ball": 0, "pingpong": 0,
    }))
    print(f"✓ Hub dataset added ({len(downloaded_dirs)} total sources)")

elif NDJSON_PATH:
    print(f"NDJSON not found: {NDJSON_PATH}")
    print("Download from platform.ultralytics.com/xi-x/datasets/ping-pong-ball → Export → NDJSON")
    print("Upload to Colab and re-run this cell.")
else:
    print("NDJSON_PATH=None — Hub dataset skipped.")

In [ ]:
def load_class_names(dataset_dir):
    with open(Path(dataset_dir) / 'data.yaml') as f:
        return yaml.safe_load(f).get('names', [])

def remap_label_file(src, dst, src_classes, class_map):
    lines_out = []
    if not src.exists():
        dst.write_text(''); return 0
    for line in src.read_text().splitlines():
        parts = line.strip().split()
        if not parts: continue
        src_id = int(parts[0])
        if src_id >= len(src_classes): continue
        name = src_classes[src_id]
        # Try: exact → lowercase → lowercase+hyphen→underscore
        tgt = class_map.get(name,
              class_map.get(name.lower(),
              class_map.get(name.lower().replace('-', '_'), -1)))
        if tgt == -1: continue
        lines_out.append(f"{tgt} {' '.join(parts[1:])}")
    dst.write_text('\n'.join(lines_out))
    return len(lines_out)

def find_split(ds_path, split):
    for name in ([split, 'val'] if split == 'valid' else [split]):
        img = ds_path / name / 'images'
        if img.exists():
            return img, ds_path / name / 'labels'
    return None, None

def merge_datasets(downloaded, out_dir):
    out = Path(out_dir)
    # Wipe before re-merge so re-runs don't accumulate stale files
    if out.exists():
        shutil.rmtree(out)
    for split in ('train', 'valid'):
        (out/split/'images').mkdir(parents=True, exist_ok=True)
        (out/split/'labels').mkdir(parents=True, exist_ok=True)
    total_imgs = total_boxes = skipped = 0
    for di, (ds_dir, cmap) in enumerate(downloaded):
        src_cls = load_class_names(ds_dir)
        print(f'  Dataset {di} ({Path(ds_dir).name}): {src_cls}')
        for split in ('train', 'valid'):
            img_src, lbl_src = find_split(Path(ds_dir), split)
            if img_src is None: continue
            for img in img_src.iterdir():
                stem = f"ds{di}_{img.stem}"
                shutil.copy2(img, out/split/'images'/(stem+img.suffix))
                n = remap_label_file(
                    lbl_src/(img.stem+'.txt'), out/split/'labels'/(stem+'.txt'),
                    src_cls, cmap)
                total_boxes += n
                if n == 0: skipped += 1
                total_imgs += 1
    with open(out/'data.yaml', 'w') as f:
        yaml.dump({'path': str(out.resolve()), 'train': 'train/images',
                   'val': 'valid/images', 'nc': len(TARGET_CLASSES),
                   'names': TARGET_CLASSES}, f, default_flow_style=False)
    print(f'\nMerge complete: {total_imgs} images, {total_boxes} annotations '
          f'({skipped} images had 0 annotations after remapping)')
    if skipped > total_imgs * 0.5:
        print('WARNING: >50% images have 0 annotations — class_map keys likely wrong.')
        print('Run the diagnostic cell above and compare names to DATASETS[i]["class_map"].')
    return str(out/'data.yaml')

DATA_YAML = merge_datasets(downloaded_dirs, '/content/unibots_merged')
print('data.yaml →', DATA_YAML)

In [ ]:
def load_class_names(dataset_dir):
    for fname in ('data.yaml', 'dataset.yaml'):
        p = Path(dataset_dir) / fname
        if p.exists():
            return yaml.safe_load(p.read_text()).get('names', [])
    return []

def remap_label_file(src, dst, src_classes, class_map):
    lines_out = []
    if not src.exists():
        dst.write_text(''); return 0
    for line in src.read_text().splitlines():
        parts = line.strip().split()
        if not parts: continue
        src_id = int(parts[0])
        if src_id >= len(src_classes): continue
        name = src_classes[src_id]
        # Try: exact → lowercase → lowercase+hyphen→underscore
        tgt = class_map.get(name,
              class_map.get(name.lower(),
              class_map.get(name.lower().replace('-', '_'), -1)))
        if tgt == -1: continue
        lines_out.append(f"{tgt} {' '.join(parts[1:])}")
    dst.write_text('\n'.join(lines_out))
    return len(lines_out)

def find_split(ds_path, split):
    # Layout 1 (Roboflow): {ds}/{split}/images/  +  {ds}/{split}/labels/
    for name in ([split, 'val'] if split == 'valid' else [split]):
        img = ds_path / name / 'images'
        if img.exists():
            return img, ds_path / name / 'labels'
    # Layout 2 (Ultralytics NDJSON): {ds}/images/{split}/  +  {ds}/labels/{split}/
    for name in ([split, 'val'] if split == 'valid' else [split]):
        img = ds_path / 'images' / name
        if img.exists():
            return img, ds_path / 'labels' / name
    return None, None

def merge_datasets(downloaded, out_dir):
    out = Path(out_dir)
    # Wipe before re-merge so re-runs don't accumulate stale files
    if out.exists():
        shutil.rmtree(out)
    for split in ('train', 'valid'):
        (out/split/'images').mkdir(parents=True, exist_ok=True)
        (out/split/'labels').mkdir(parents=True, exist_ok=True)
    total_imgs = total_boxes = skipped = 0
    for di, (ds_dir, cmap) in enumerate(downloaded):
        src_cls = load_class_names(ds_dir)
        print(f'  Dataset {di} ({Path(ds_dir).name}): {src_cls}')
        for split in ('train', 'valid'):
            img_src, lbl_src = find_split(Path(ds_dir), split)
            if img_src is None: continue
            for img in img_src.iterdir():
                stem = f"ds{di}_{img.stem}"
                shutil.copy2(img, out/split/'images'/(stem+img.suffix))
                n = remap_label_file(
                    lbl_src/(img.stem+'.txt'), out/split/'labels'/(stem+'.txt'),
                    src_cls, cmap)
                total_boxes += n
                if n == 0: skipped += 1
                total_imgs += 1
    with open(out/'data.yaml', 'w') as f:
        yaml.dump({'path': str(out.resolve()), 'train': 'train/images',
                   'val': 'valid/images', 'nc': len(TARGET_CLASSES),
                   'names': TARGET_CLASSES}, f, default_flow_style=False)
    print(f'\nMerge complete: {total_imgs} images, {total_boxes} annotations '
          f'({skipped} images had 0 annotations after remapping)')
    if skipped > total_imgs * 0.5:
        print('WARNING: >50% images have 0 annotations — class_map keys likely wrong.')
        print('Run the diagnostic cell above and compare names to DATASETS[i]["class_map"].')
    return str(out/'data.yaml')

DATA_YAML = merge_datasets(downloaded_dirs, '/content/unibots_merged')
print('data.yaml →', DATA_YAML)

## 6. Synthetic Arena Image Generation

**CPU-only, no GPU needed.** ~2–5 ms/image — runs on RPi4, Banana Pi, NanoPi, anything.

Encodes **Unibots 2026 V1.2 rulebook** exactly:

| Element | Rulebook spec | Rendered as |
|---|---|---|
| Arena floor | White painted (§4.1.3) | White with slight brightness variation |
| Tape lines | 19 mm black, 4 rounded-square tracks (§4.4) | Black lines/curves, straight + curved |
| Walls | Black, ~150 mm (§4.1.2) | Dark band at frame edge |
| Scoring zones | Yellow/Orange/Purple/Green (§4.1.2) | Colored band at frame edge |
| Ping pong | 40 mm, orange, may have markings (§4.3.2) | Orange sphere, Phong shading, optional seam |
| Bearing | 20 mm, polished carbon steel (§4.3.3) | Grey sphere, high specular highlight |
| Ball ratio | 24 bearings : 16 ping pong (§4.3.1) | 60 % bearing, 40 % ping pong |
| Detection range | 15–130 cm (from competitor code) | radius 8–80 px ping pong, 4–40 px bearing |
| Edge balls | 35 % of balls at frame boundary | Fixes edge detection weakness |

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
import random, math
from pathlib import Path
import cv2

# ── Rulebook §4.1.2: scoring zone wall colours (BGR) ─────────────────────────
SCORING_ZONE_COLORS_BGR = {
    'Yellow': (0,   215, 230),
    'Orange': (0,   140, 255),
    'Purple': (160,  50, 140),
    'Green':  (40,  180,  50),
}


def generate_arena_background(width: int = 640, height: int = 480) -> np.ndarray:
    """
    Rulebook §4.1.3: arena floor is WHITE painted.
    Adds black 19 mm tape lines (§4.4) and optional wall/scoring-zone band (§4.1.2).
    Returns H×W×3 uint8 BGR.
    """
    # ── White floor with subtle brightness variation ───────────────────────────
    # White floor reflects overhead LEDs: centre often brighter than edges
    base_v = random.randint(225, 250)
    noise  = np.random.normal(base_v, random.uniform(3, 10), (height, width, 3))

    # Radial brightness gradient (overhead light)
    if random.random() < 0.7:
        cx = random.uniform(0.2*width, 0.8*width)
        cy = random.uniform(0.2*height, 0.8*height)
        xs, ys = np.meshgrid(np.arange(width), np.arange(height))
        d = np.sqrt((xs-cx)**2 + (ys-cy)**2) / math.sqrt(width**2+height**2)
        gradient = 1.0 - d * random.uniform(0.1, 0.3)
        noise *= gradient[:, :, np.newaxis]

    # Occasional shadow patch (robot arm, other robot nearby)
    if random.random() < 0.3:
        sx = random.randint(0, width);   sy = random.randint(0, height)
        sr = random.randint(40, 150)
        xs, ys = np.meshgrid(np.arange(width), np.arange(height))
        mask = np.exp(-((xs-sx)**2+(ys-sy)**2)/(2*sr**2))
        noise -= mask[:,:,np.newaxis] * random.uniform(15, 50)

    arr = noise.clip(180, 255).astype(np.uint8)  # white floor: never goes dark
    img = Image.fromarray(arr[:,:,::-1])          # BGR→RGB for PIL
    draw = ImageDraw.Draw(img)

    # ── Black 19 mm tape lines (§4.4, rounded-square tracks) ─────────────────
    # Scale 19 mm to pixels: at 640 px wide covering ~half the 2000 mm arena,
    # 1 mm ≈ 0.32 px → 19 mm ≈ 6 px. Range 4–12 depending on camera distance.
    tape_w = random.randint(4, 12)
    tape_col = (15, 15, 15)

    tape_r = random.random()
    if tape_r < 0.25:
        # Straight horizontal segment
        y = random.randint(height//6, 5*height//6)
        draw.rectangle([0, y-tape_w//2, width, y+tape_w//2], fill=tape_col)
    elif tape_r < 0.50:
        # Straight vertical segment
        x = random.randint(width//6, 5*width//6)
        draw.rectangle([x-tape_w//2, 0, x+tape_w//2, height], fill=tape_col)
    elif tape_r < 0.70:
        # Diagonal / angled line (camera tilt or corner approach)
        x1 = random.randint(0, width//2);   y1 = random.randint(0, height)
        x2 = random.randint(width//2, width); y2 = random.randint(0, height)
        draw.line([x1,y1,x2,y2], fill=tape_col, width=tape_w)
    elif tape_r < 0.85:
        # Arc / curved track corner
        ocx = random.randint(-width//3, 4*width//3)
        ocy = random.randint(-height//3, 4*height//3)
        rad = random.randint(max(width,height)//4, max(width,height))
        a1  = random.randint(0, 270)
        draw.arc([ocx-rad,ocy-rad,ocx+rad,ocy+rad], a1, a1+90,
                 fill=tape_col, width=tape_w)
    # else: no tape visible (~15 % of images)

    # ── Arena wall / scoring zone at frame edge (§4.1.2) ─────────────────────
    if random.random() < 0.40:
        wall_h = random.randint(8, 55)   # px visible at top of frame
        draw.rectangle([0, 0, width, wall_h], fill=(25, 25, 25))  # black wall

        # Scoring zone section on that wall (~50 % chance)
        if random.random() < 0.55:
            col_bgr = random.choice(list(SCORING_ZONE_COLORS_BGR.values()))
            col_rgb = (col_bgr[2], col_bgr[1], col_bgr[0])
            zx = random.randint(0, width//2)
            zw = random.randint(width//5, width//2)
            draw.rectangle([zx, 0, zx+zw, wall_h], fill=col_rgb)

    return np.array(img)[:,:,::-1]  # RGB→BGR


# ── Ball renderers ─────────────────────────────────────────────────────────────

def _render_sphere(radius_px: int, base_rgb: tuple, spec_rgb: tuple,
                   shadow_str: float = 0.5) -> np.ndarray:
    """Returns (2R×2R, 4) RGBA uint8 sphere with Phong shading."""
    size = radius_px * 2
    img  = np.zeros((size, size, 4), dtype=np.float32)
    lx, ly, lz = -0.6, -0.6, 0.5                    # upper-left overhead light
    ln = math.sqrt(lx*lx + ly*ly + lz*lz)
    lx, ly, lz = lx/ln, ly/ln, lz/ln
    for y in range(size):
        for x in range(size):
            dx = (x - radius_px) / radius_px
            dy = (y - radius_px) / radius_px
            r2 = dx*dx + dy*dy
            if r2 > 1.0: continue
            nz      = math.sqrt(max(0.0, 1.0-r2))
            diffuse = max(0.0, dx*lx + dy*ly + nz*lz)
            spec    = max(0.0, diffuse)**20
            t       = 0.25 + 0.75*diffuse
            for c in range(3):
                col = base_rgb[c]/255.0 * t * (1-shadow_str*(1-t))
                col = min(1.0, col + spec * spec_rgb[c]/255.0)
                img[y, x, c] = col*255
            edge = max(0.0, 1.0 - max(0.0, r2-0.85)/0.15)
            img[y, x, 3] = min(255.0, edge*255)
    return img.astype(np.uint8)


def render_ping_pong_ball(radius_px: int) -> np.ndarray:
    """
    §4.3.2: orange, 40 mm, may have markings.
    15 % chance of white variant (some sets include white balls).
    """
    if random.random() < 0.15:                    # white variant
        base = (random.randint(230,255),)*3
    else:                                          # orange (HSV H 5-25°)
        base = (random.randint(220,255),
                random.randint(85,155),
                random.randint(0,35))
    ball = _render_sphere(radius_px, base, (255,255,245), shadow_str=0.35)

    # Optional seam marking (§4.3.2: "may have markings")
    if random.random() < 0.4 and radius_px >= 10:
        img_pil = Image.fromarray(ball, 'RGBA')
        draw    = ImageDraw.Draw(img_pil)
        size    = radius_px*2
        offset  = random.randint(-radius_px//3, radius_px//3)
        seam_col = (max(0,base[0]-60), max(0,base[1]-40), max(0,base[2]-20), 200)
        draw.arc([size//4, size//4+offset, 3*size//4, 3*size//4+offset],
                 0, 180, fill=seam_col, width=max(1, radius_px//12))
        ball = np.array(img_pil)

    return ball


def render_bearing(radius_px: int) -> np.ndarray:
    """
    §4.3.3: 20 mm polished carbon steel bearing.
    Characteristic: very bright specular highlight on dark grey base.
    Saturation < 70 (consistent with competitor HSV detection sat < 70).
    """
    v     = random.randint(55, 130)             # medium-dark grey base
    tint  = random.randint(-5, 15)              # slight steel-blue tint
    base  = (max(0,v-8), max(0,v+3), max(0,v+tint))  # R, G, B (slight cool tint)
    return _render_sphere(radius_px, base, (255,255,255), shadow_str=0.75)


BALL_RENDERERS = {0: render_ping_pong_ball, 1: render_bearing}

print('Synthetic generators loaded.')

In [ ]:
def paste_ball_rgba(bg_bgr: np.ndarray, ball_rgba: np.ndarray, cx: int, cy: int):
    h, w   = bg_bgr.shape[:2]
    bh, bw = ball_rgba.shape[:2]
    x1,y1  = cx-bw//2, cy-bh//2
    sx1 = max(0,-x1);  sy1 = max(0,-y1)
    sx2 = bw-max(0,x1+bw-w); sy2 = bh-max(0,y1+bh-h)
    dx1 = max(0,x1);   dy1 = max(0,y1)
    dx2 = min(w,x1+bw); dy2 = min(h,y1+bh)
    if dx2<=dx1 or dy2<=dy1: return bg_bgr
    crop  = ball_rgba[sy1:sy2, sx1:sx2]
    alpha = crop[:,:,3:4].astype(np.float32)/255.0
    bball = crop[:,:,:3][:,:,::-1].astype(np.float32)  # RGBA RGB→BGR
    out   = bg_bgr.copy().astype(np.float32)
    out[dy1:dy2,dx1:dx2] = alpha*bball + (1-alpha)*out[dy1:dy2,dx1:dx2]
    return out.clip(0,255).astype(np.uint8)


def generate_synthetic_image(img_w=640, img_h=480, n_balls_range=(1,6)):
    """
    Returns (bgr_ndarray, list_of_yolo_label_strings).

    Ball class ratio 3:2 bearing:pingpong matches §4.3.1 (24 bearings : 16 pingpong).
    35 % of balls deliberately placed at/near frame edges to train edge detection.

    Radius ranges encode §4.3 physical sizes at 15–130 cm detection range
    (focal_px ≈ 554 at 60° HFOV, 640 px wide):
      ping_pong 40 mm → ~8–80 px radius
      bearing   20 mm → ~4–40 px radius
    """
    bg = generate_arena_background(img_w, img_h)
    labels = []

    for _ in range(random.randint(*n_balls_range)):
        # §4.3.1 ratio: 24 bearing : 16 ping_pong = 3:2
        cls_id = random.choice([0, 0, 1, 1, 1])

        if cls_id == 0:   # ping_pong_ball, 40 mm
            radius_px = random.randint(8, 80)
        else:             # bearing, 20 mm (half the diameter → half the radius at same dist)
            radius_px = random.randint(4, 40)

        # 35 % edge placement (fixes edge detection weakness)
        if random.random() < 0.35:
            edge = random.choice(['L','R','T','B'])
            if   edge=='L': cx=random.randint(-radius_px//2, radius_px);    cy=random.randint(0,img_h)
            elif edge=='R': cx=random.randint(img_w-radius_px,img_w+radius_px//2); cy=random.randint(0,img_h)
            elif edge=='T': cx=random.randint(0,img_w); cy=random.randint(-radius_px//2,radius_px)
            else:           cx=random.randint(0,img_w); cy=random.randint(img_h-radius_px,img_h+radius_px//2)
        else:
            cx = random.randint(radius_px, img_w-radius_px)
            cy = random.randint(radius_px, img_h-radius_px)

        ball_rgba = BALL_RENDERERS[cls_id](radius_px)
        bg = paste_ball_rgba(bg, ball_rgba, cx, cy)

        # YOLO label — clamp bbox to image
        x1 = max(0, cx-radius_px)/img_w;  x2 = min(img_w, cx+radius_px)/img_w
        y1 = max(0, cy-radius_px)/img_h;  y2 = min(img_h, cy+radius_px)/img_h
        bw, bh = x2-x1, y2-y1
        if bw > 0.005 and bh > 0.005:
            labels.append(f"{cls_id} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} {bw:.6f} {bh:.6f}")

    return bg, labels


print('Compositor ready.')

In [ ]:
# Visual preview — verify white floor, black lines, ball colours
from IPython.display import Image as IPImage, display

COLORS_BGR = [(0,130,255), (160,160,160), (0,0,220)]
previews = []
for _ in range(3):
    bgr, labs = generate_synthetic_image(640, 480, n_balls_range=(3,8))
    vis = bgr.copy()
    for lab in labs:
        c, cx_, cy_, bw_, bh_ = int(lab.split()[0]), *map(float, lab.split()[1:])
        x1=int((cx_-bw_/2)*640); y1=int((cy_-bh_/2)*480)
        x2=int((cx_+bw_/2)*640); y2=int((cy_+bh_/2)*480)
        cv2.rectangle(vis,(x1,y1),(x2,y2),COLORS_BGR[c],2)
        cv2.putText(vis,TARGET_CLASSES[c],(x1,max(y1-4,10)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.4,COLORS_BGR[c],1)
    previews.append(vis)

mosaic = np.hstack(previews)
cv2.imwrite('/tmp/synthetic_preview.jpg', mosaic)
display(IPImage('/tmp/synthetic_preview.jpg'))
print('Check: white floor, black tape lines, orange ping pong, grey bearings')

In [ ]:
import time

syn_img = Path('/content/unibots_merged/train/images')
syn_lbl = Path('/content/unibots_merged/train/labels')

t0 = time.perf_counter()
for i in range(N_SYNTHETIC_IMAGES):
    bgr, labs = generate_synthetic_image(640, 480, n_balls_range=(1,7))
    stem = f"syn_{i:05d}"
    cv2.imwrite(str(syn_img/(stem+'.jpg')), bgr, [cv2.IMWRITE_JPEG_QUALITY,92])
    (syn_lbl/(stem+'.txt')).write_text('\n'.join(labs))

elapsed = time.perf_counter()-t0
print(f"Generated {N_SYNTHETIC_IMAGES} images in {elapsed:.1f}s "
      f"({elapsed/N_SYNTHETIC_IMAGES*1000:.1f} ms/image)")
print(f"Total training images: {len(list(syn_img.iterdir()))}")

## 7. Train YOLO11n

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data=DATA_YAML,
    epochs=TRAIN_EPOCHS,
    imgsz=TRAIN_IMGSZ,
    batch=TRAIN_BATCH,
    device=0,
    workers=4,
    project='/content/runs',
    name='unibots_v1',
    exist_ok=True,
    cache=True,

    # ── Augmentation: edge detection + scale + white-floor robustness ─────────
    mosaic=1.0,
    copy_paste=0.3,      # places objects at frame edges
    scale=0.7,           # 15–130 cm range = big size variation
    translate=0.15,      # shifts balls toward frame border
    degrees=10.0,        # camera tilt
    flipud=0.1,
    fliplr=0.5,
    hsv_h=0.02,
    hsv_s=0.9,           # orange ping pong vs white variant vs grey bearing
    hsv_v=0.5,           # white floor reflection vs shadows
    erasing=0.4,         # partial occlusion
    perspective=0.0005,
    close_mosaic=15,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    patience=30,
    label_smoothing=0.1,
)
print('Training complete:', results.save_dir)

## 8. Evaluate

In [ ]:
from IPython.display import Image as IPImage, display
run_dir = Path('/content/runs/unibots_v1')
for png in ['results.png','confusion_matrix_normalized.png','val_batch0_pred.jpg']:
    p = run_dir/png
    if p.exists(): print(png); display(IPImage(str(p)))

In [ ]:
best_pt = Path('/content/runs/unibots_v1/weights/best.pt')
metrics = YOLO(str(best_pt)).val(data=DATA_YAML, imgsz=TRAIN_IMGSZ, device=0)
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print('Per-class AP50:', dict(zip(TARGET_CLASSES, metrics.box.ap50)))

## 9. Export to NCNN

In [ ]:
import shutil
best_pt = Path('/content/runs/unibots_v1/weights/best.pt')
out_dir = Path('/content/ncnn_models'); out_dir.mkdir(exist_ok=True)

def export_ncnn(pt, imgsz, suffix):
    m = YOLO(str(pt))
    m.export(format='ncnn', imgsz=imgsz, half=False)
    nd = pt.parent/'best_ncnn_model'
    pd = out_dir/f'model_{suffix}.ncnn.param'
    bd = out_dir/f'model_{suffix}.ncnn.bin'
    shutil.copy2(nd/'model.ncnn.param', pd)
    shutil.copy2(nd/'model.ncnn.bin',   bd)
    return pd, bd

print(f'Exporting {EXPORT_IMGSZ_PRIMARY}×{EXPORT_IMGSZ_PRIMARY} (30fps target)...')
p256, b256 = export_ncnn(best_pt, EXPORT_IMGSZ_PRIMARY, EXPORT_IMGSZ_PRIMARY)
print(f'Exporting {EXPORT_IMGSZ_SECONDARY}×{EXPORT_IMGSZ_SECONDARY} (accuracy)...')
p320, b320 = export_ncnn(best_pt, EXPORT_IMGSZ_SECONDARY, EXPORT_IMGSZ_SECONDARY)

text = p256.read_text()
print(f'Layer check — in0: {"in0" in text}, out0: {"out0" in text}')

## 10. Benchmark (RPi4 estimate)

In [ ]:
import time
try:
    import ncnn
    def bench(param, bin_, imgsz, n=40):
        net = ncnn.Net()
        net.opt.num_threads=4; net.opt.use_fp16_packed=True
        net.opt.use_fp16_storage=True; net.opt.use_bf16_storage=True
        net.opt.use_winograd_convolution=True; net.opt.use_sgemm_convolution=True
        net.load_param(str(param)); net.load_model(str(bin_))
        dummy=ncnn.Mat.from_pixels(np.random.randint(0,255,(imgsz,imgsz,3),dtype=np.uint8),
                                   ncnn.Mat.PixelType.PIXEL_BGR2RGB,imgsz,imgsz)
        dummy.substract_mean_normalize(None,(1/255.,)*3)
        ts=[]
        for _ in range(n):
            ex=net.create_extractor(); ex.input('in0',dummy)
            t0=time.perf_counter(); ex.extract('out0',ncnn.Mat())
            ts.append((time.perf_counter()-t0)*1000)
        return float(np.median(ts))
    for imgsz,p,b in [(EXPORT_IMGSZ_PRIMARY,p256,b256),(EXPORT_IMGSZ_SECONDARY,p320,b320)]:
        ms=bench(p,b,imgsz); rpi=ms*3.5
        print(f'{imgsz}×{imgsz}: Colab {ms:.0f}ms | RPi4/BPi est. {rpi:.0f}ms = {1000/rpi:.0f}fps')
except ImportError:
    print('ncnn Python not on Colab — reference: YOLO11n@256 ≈ 30fps, @320 ≈ 22fps on RPi4')

## 11. Package & Download

In [ ]:
import zipfile
zip_path='/content/unibots_models.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    zf.write(p256,'model_256/model.ncnn.param')
    zf.write(b256,'model_256/model.ncnn.bin')
    zf.write(p320,'model_320/model.ncnn.param')
    zf.write(b320,'model_320/model.ncnn.bin')
    if best_pt.exists(): zf.write(best_pt,'best.pt')
print(f'Packaged: {zip_path} ({Path(zip_path).stat().st_size/1e6:.1f} MB)')

In [ ]:
from google.colab import files
files.download('/content/unibots_models.zip')

## 12. Deployment

### Copy model files
```bash
# Primary: ~30fps on RPi4 / Banana Pi / NanoPi (4 threads)
cp model_256/model.ncnn.param  unibots_ws/src/unibots_perception/models/model.ncnn.param
cp model_256/model.ncnn.bin    unibots_ws/src/unibots_perception/models/model.ncnn.bin
cd unibots_ws && colcon build --packages-select unibots_perception --symlink-install
source install/setup.bash
ros2 run unibots_perception perception_node --ros-args \
  -p input_size:=256 -p num_threads:=4 -p hfov_deg:=YOUR_CALIBRATED_VALUE
```

### ⚠️ Calibrate `hfov_deg` before competition (§4.1.3 arena, §4.3.2 ball)
1. Place a ping pong ball (40 mm, §4.3.2) at exactly **1000 mm** from camera
2. Measure pixel width `px_w` in a debug frame
3. `focal_px = (1000 × px_w) / 40`
4. `hfov_deg = 2 × atan(img_width / (2 × focal_px)) × (180/π)`
5. `ros2 param set /perception_node hfov_deg <value>`

### ⚠️ Bearing diameter is 20 mm (§4.3.3) — already set in perception_node.cpp
If competition organiser changes spec, update `BEARING_DIAMETER_MM` in `perception_node.cpp`.

### To improve further
- Increase `N_SYNTHETIC_IMAGES` to 3000–5000
- Capture real images on the actual competition arena and label them → add as a 5th Roboflow dataset
- Run `debug_visualiser` at a practice session and save edge-case frames